In [24]:
##rag pipelines -Data Ingestion to vector db pipeline
import os
from langchain_community.document_loaders import PyMuPDFLoader,PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [25]:
##read all the pdfs
from pathlib import Path


def process_all_pdfs(pdf_directory):
    """Process all PDF files in the specified directory and return a list of documents."""
    all_documents = []
    pdf_dir=Path(pdf_directory)
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print(f"Found {len(pdf_files)} PDF files in directory: {pdf_directory}")
    for pdf_file in pdf_files:
        loader = PyPDFLoader(str(pdf_file))
        documents = loader.load()
        for doc in documents:
            doc.metadata["source_file"] = pdf_file.name
            doc.metadata['file_type']='pdf'
        all_documents.extend(documents)
    return all_documents

In [26]:
all_documents = process_all_pdfs("../data/pdf_files")

Found 7 PDF files in directory: ../data/pdf_files


In [27]:
all_documents 

[Document(metadata={'producer': 'WeasyPrint 69.0', 'creator': 'PyPDF', 'creationdate': '', 'source': '../data/pdf_files/Mayank_Bisht_Resume.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Mayank_Bisht_Resume.pdf', 'file_type': 'pdf'}, page_content='MAYANK BISHT+91 8076405033 \xa0|\xa0 mayankbisht1107@gmail.com \xa0|\xa0 Delhi, India\nGitHub: github.com/mayankbisht-tech \xa0|\xa0 LinkedIn: linkedin.com/in/mayank-bisht-a68491325\nPROFESSIONAL SUMMARY\nFull-Stack  Web  Development  intern  candidate  with  hands-on  experience  designing  responsive  front-end\ninterfaces in React, Next.js, and Tailwind CSS, and building back-end services and REST APIs with Node.js\nand Express.js. Comfortable designing and querying both relational and non-relational databases — MySQL,\nPostgreSQL, and MongoDB — and debugging full-stack issues using Git and Postman. Strong fundamentals in\nDSA, OOP , and DBMS, with a track record of shipping complete features independently. Additiona

In [28]:
###text splitter
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks using RecursiveCharacterTextSplitter."""
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap,length_function=len,separators=["\n\n", "\n", " ", ""])
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks.")
    if split_docs:
        print(f"Example chunk ")
        print(f"Example chunk content: {split_docs[0].page_content[:200]}...")    
        print(f"Example chunk metadata: {split_docs[0].metadata}")
    return split_docs   


In [29]:
split_docs = split_documents(all_documents)

Split 26 documents into 77 chunks.
Example chunk 
Example chunk content: MAYANK BISHT+91 8076405033  |  mayankbisht1107@gmail.com  |  Delhi, India
GitHub: github.com/mayankbisht-tech  |  LinkedIn: linkedin.com/in/mayank-bisht-a68491325
PROFESSIONAL SUMMARY
Full-Stack  Web ...
Example chunk metadata: {'producer': 'WeasyPrint 69.0', 'creator': 'PyPDF', 'creationdate': '', 'source': '../data/pdf_files/Mayank_Bisht_Resume.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Mayank_Bisht_Resume.pdf', 'file_type': 'pdf'}


In [30]:
###embedding and vector store
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List,Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [31]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer."""
    def __init__(self,model_name:str="all-MiniLM-L6-v2"):
        """"
        Initialize the EmbeddingManager with a specified model name.
        
        Args:
            model_name (str): The name of the SentenceTransformer model to use for embeddings.
        """
        self.model_name = model_name
        self.model=None
        self._load_model()
    def _load_model(self):
        """Load the SentenceTransformer model."""
        self.model = SentenceTransformer(self.model_name)
        print(f"Loaded embedding model: {self.model_name},embedding dimension: {self.model.get_sentence_embedding_dimension()}")
    def generate_embeddings(self, texts:List[str])->np.ndarray:
        """Generate embeddings for a list of texts.
        
        Args:
            texts (List[str]): A list of text strings to embed.
        
        Returns:
            np.ndarray: An array of embeddings for the input texts.
        """
        if self.model is None:
            self._load_model()
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts,show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
    def get_embedding_dimension(self)->int:
        """Get the dimension of the embeddings produced by the model.
        
        Returns:
            int: The dimension of the embeddings.
        """
        if self.model is None:
            self._load_model()
        return self.model.get_sentence_embedding_dimension()
embedding_manager = EmbeddingManager(model_name="all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13019.12it/s]


Loaded embedding model: all-MiniLM-L6-v2,embedding dimension: 384


/tmp/ipykernel_3115/1211826971.py:16: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Loaded embedding model: {self.model_name},embedding dimension: {self.model.get_sentence_embedding_dimension()}")


In [32]:
##vector store
class VectorStore:
    """Handles storage and retrieval of document embeddings using ChromaDB."""
    def __init__(self,collection_name:str="pdf_document",persist_directory:str="../data/chroma_db"):
        """
        Initialize the VectorStore with specified parameters.
        
        Args:
            collection_name (str): The name of the ChromaDB collection.
            persist_directory (str): The directory to persist the ChromaDB data.
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client=None
        self.collection=None
        self._initialize_chromadb()
    def _initialize_chromadb(self):
        """Initialize the ChromaDB client and collection."""
        os.makedirs(self.persist_directory, exist_ok=True)
        self.client = chromadb.PersistentClient(path=self.persist_directory)
        self.collection=self.client.get_or_create_collection(name=self.collection_name,metadata={"description": "Collection of PDF document embeddings"})
        print(f"Initialized ChromaDB collection: {self.collection_name} at {self.persist_directory}")
        print(f"Existing items in collection: {self.collection.count()}")
    def add_document(self,documents:List[Any],embeddings:np.ndarray):
        """Add documents and their embeddings to the vector store.
        
        Args:
            documents (List[Dict[str, Any]]): A list of documents with 'id', 'embedding', and 'metadata'.
            embeddings (np.ndarray): An array of embeddings corresponding to the documents.
        """
        if len(documents) != len(embeddings):
            raise ValueError("The number of documents must match the number of embeddings.")
        print(f"Adding {len(documents)} documents to collection: {self.collection_name}")
        ids=[]
        metadatas=[]
        documents_texts=[]
        embeddings_list=[]
        for i,(doc,embedding) in enumerate(zip(documents,embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            metadata=dict(doc.metadata)
            metadata['doc_index']=i
            metadata["content_length"]=len(doc.page_content)

            metadatas.append(metadata)
            documents_texts.append(doc.page_content)
            embeddings_list.append(embedding.tolist())

        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_texts
            )
            print(f"Successfully added {len(documents)} documents to collection: {self.collection_name}")   
        except Exception as e:
            print(f"Error adding documents to collection: {e}")

In [33]:
vector_store = VectorStore(collection_name="pdf_document", persist_directory="../data/chroma_db")

Initialized ChromaDB collection: pdf_document at ../data/chroma_db
Existing items in collection: 28


In [34]:
split_docs

[Document(metadata={'producer': 'WeasyPrint 69.0', 'creator': 'PyPDF', 'creationdate': '', 'source': '../data/pdf_files/Mayank_Bisht_Resume.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Mayank_Bisht_Resume.pdf', 'file_type': 'pdf'}, page_content='MAYANK BISHT+91 8076405033 \xa0|\xa0 mayankbisht1107@gmail.com \xa0|\xa0 Delhi, India\nGitHub: github.com/mayankbisht-tech \xa0|\xa0 LinkedIn: linkedin.com/in/mayank-bisht-a68491325\nPROFESSIONAL SUMMARY\nFull-Stack  Web  Development  intern  candidate  with  hands-on  experience  designing  responsive  front-end\ninterfaces in React, Next.js, and Tailwind CSS, and building back-end services and REST APIs with Node.js\nand Express.js. Comfortable designing and querying both relational and non-relational databases — MySQL,\nPostgreSQL, and MongoDB — and debugging full-stack issues using Git and Postman. Strong fundamentals in\nDSA, OOP , and DBMS, with a track record of shipping complete features independently. Additiona

In [35]:
texts=[doc.page_content for doc in split_docs]  

In [36]:
embeddings = embedding_manager.generate_embeddings(texts)

Generating embeddings for 77 texts...


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Batches: 100%|██████████| 3/3 [00:06<00:00,  2.23s/it]

Generated embeddings with shape: (77, 384)


In [37]:
vector_store.add_document(split_docs,embeddings)

Adding 77 documents to collection: pdf_document
Successfully added 77 documents to collection: pdf_document


In [52]:
##reteriver pipeline from vector store
class RAGReteriver:
    """Implements a Retrieval-Augmented Generation (RAG) retriever using a vector store."""
    def __init__(self,vector_store:VectorStore,embedding_manager:EmbeddingManager,top_k:int=5):
        """
        Initialize the RAGReteriver with a vector store, embedding manager, and top_k parameter.
        
        Args:
            vector_store (VectorStore): An instance of the VectorStore class for document retrieval.
            embedding_manager (EmbeddingManager): An instance of the EmbeddingManager class for generating query embeddings.
            top_k (int): The number of top similar documents to retrieve for a given query.
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager
        self.top_k = top_k
    def retrieve(self, query:str,top_k:int=5,score_threshold:float=0.0)->List[Dict[str,Any]]:
        """Retrieve the top_k most similar documents for a given query.
        
        Args:
            query (str): The input query string.
            top_k (int): The number of top similar documents to retrieve.
            score_threshold (float): The minimum similarity score threshold for retrieved documents.

        returns:
            List[Dict[str, Any]]: A list of dictionaries containing the retrieved documents and their metadata.
        """
        print(f"Retrieving documents for query: '{query}' with top_k={top_k} and score_threshold={score_threshold}")
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k,
                include=["documents", "metadatas", "distances"]
            )
            
            print("Distances:", results["distances"][0])
            print("Documents:", len(results["documents"][0]))
            retrieved_docs=[]
            if results["documents"] and results['documents'][0]:
                documents=results['documents'][0]
                metadatas=results['metadatas'][0]
                distances=results['distances'][0]
                ids=results['ids'][0]
                for i,(doc_id,doc,metadata,distance) in enumerate(zip(ids,documents,metadatas,distances)):
                    retrieved_docs.append({
                        "id": doc_id,
                        "document": doc,
                        "metadata": metadata,
                        "distance":distance,
                        "rank": i + 1
                    })
            print(f"Retrieved {len(retrieved_docs)} documents above the score threshold of {score_threshold}.")
            return retrieved_docs
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []   


In [39]:
rag_retriever = RAGReteriver(vector_store=vector_store, embedding_manager=embedding_manager, top_k=5)

In [56]:
print(
    rag_retriever.retrieve(
        "Encoder",
        top_k=3,
    )
)

Retrieving documents for query: 'Encoder' with top_k=3 and score_threshold=0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 76.12it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents above the score threshold of 0.0.
[{'id': 'doc_f91c8bc1_40', 'document': 'connected layers for both the encoder and decoder, shown in the left and right halves of Figure 1,\nrespectively.\n3.1 Encoder and Decoder Stacks\nEncoder: The encoder is composed of a stack of N = 6 identical layers. Each layer has two\nsub-layers. The ﬁrst is a multi-head self-attention mechanism, and the second is a simple, position-\n2', 'metadata': {'subject': 'Neural Information Processing Systems http://nips.cc/', 'language': 'en-US', 'title': 'Attention is All you Need', 'created': '2017', 'source_file': 'NIPS-2017-attention-is-all-you-need-Paper.pdf', 'editors': 'I. Guyon and U.V. Luxburg and S. Bengio and H. Wallach and R. Fergus and S. Vishwanathan and R. Garnett', 'file_type': 'pdf', 'firstpage': '5998', 'moddate': '2018-02-12T21:22:10-08:00', 'date': '2017', 'doc_index': 40, 'author': 'Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Usz

In [57]:
##integration vectordb context with llm output
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv()
import os
model=init_chat_model("groq:qwen/qwen3-32b")

In [58]:
def rag_simple(query:str,rag_retriever:RAGReteriver,model,top_k:int=5,)->str:
    """Perform a simple Retrieval-Augmented Generation (RAG) process for a given query.
    
    Args:
        query (str): The input query string.
        rag_retriever (RAGReteriver): An instance of the RAGReteriver class for document retrieval.
        model: A language model instance for generating responses based on retrieved documents.
        top_k (int): The number of top similar documents to retrieve for the query.
        score_threshold (float): The minimum similarity score threshold for retrieved documents.

    Returns:
        str: The generated response based on the retrieved documents and the input query.
    """
    retrieved_docs = rag_retriever.retrieve(query=query, top_k=top_k)
    if not retrieved_docs:
        return "No relevant documents found for the query."
    
    context = "\n\n".join([f"Document {doc['rank']} (Score: {doc['similarity_score']:.4f}):\n{doc['document']}" for doc in retrieved_docs])
    
    prompt = f"Use the following retrieved documents to answer the question:\n\n{context}\n\nQuestion: {query}\nAnswer:"
    
    response = model.invoke([prompt.format(context=context, query=query)])
    
    return response.content

In [60]:
answer=rag_simple("Encoder", rag_retriever, model, top_k=1)
print("Generated Answer:")
print(answer)

Retrieving documents for query: 'Encoder' with top_k=1 and score_threshold=0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 33.66it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents above the score threshold of 0.0.


Generated Answer:
<think>
Okay, let's tackle this question. The user is asking about the encoder, so I need to refer to the provided document.

Looking at Document 1, it mentions that the encoder is composed of a stack of N = 6 identical layers. Each layer has two sub-layers: the first is a multi-head self-attention mechanism, and the second is a simple, position-wise sub-layer. Wait, the second part is cut off in the document, but based on common transformer architecture knowledge, the second sub-layer is usually a position-wise fully connected feed-forward network. The score here is low (0.0928), which might mean the document isn't very relevant, but since it's the only one provided, I'll go with that.

I should structure the answer to first state the number of layers (6), then each layer's components: multi-head self-attention and position-wise feed-forward network. Since the document mentions "position-" but gets cut off, I'll check if there's any more info. The next line in the do